In [1]:
import pandas as pd
from tqdm import tqdm
from datetime import datetime, timezone
import requests
import time
import numpy as np

/Users/itsgil/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
API_KEY = "#"

def utc_to_local_str(utc_str):
    try:
        utc_dt = pd.to_datetime(utc_str, utc=True)
        local_dt = utc_dt.tz_convert(None)
        return local_dt.strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return None

# Fetch inradius history data for a given lat/lon/radius and date range
def fetch_inradius_history(lat, lon, radius, from_date, to_date):
    from_str = pd.to_datetime(from_date).strftime('%Y-%m-%d')
    to_str = pd.to_datetime(to_date).strftime('%Y-%m-%d')
    
    url = (f"https://api.datalastic.com/api/v0/report?"
           f"api-key={API_KEY}&report_type=inradius_history"
           f"&lat={lat}&lon={lon}&radius={radius}"
           f"&from={from_str}&to={to_str}")
    try:
        resp = requests.get(url)
        resp.raise_for_status()
        data = resp.json()
        return data.get("data", [])
    except Exception as e:
        print(f"Error fetching inradius history: {e}")
        return []

def fetch_positions(uuid, from_date, months_offset):
    start = pd.to_datetime(from_date) + pd.DateOffset(months=months_offset)
    end = start + pd.DateOffset(months=1)
    from_str = start.strftime('%Y-%m-%d')
    to_str = end.strftime('%Y-%m-%d')
    url = f"https://api.datalastic.com/api/v0/vessel_history?api-key={API_KEY}&uuid={uuid}&from={from_str}&to={to_str}"
    try:
        resp = requests.get(url)
        resp.raise_for_status()
        data = resp.json()
        return data.get("data", {}).get("positions", [])
    except Exception as e:
        print(f"Error for uuid {uuid}: {e}")
        return []

def fetch_vessel_info(uuid):
    url = f"https://api.datalastic.com/api/v0/vessel_info?api-key={API_KEY}&uuid={uuid}"
    try:
        resp = requests.get(url)
        resp.raise_for_status()
        data = resp.json().get("data", {})
        return data
    except Exception as e:
        print(f"Error for uuid {uuid}: {e}")
        return {}

In [ ]:
# Fetch inradius history for Brunswick area
brunswick_data = fetch_inradius_history(
    lat=31.1291,
    lon=-81.544,
    radius=10,
    from_date="2025-07-31",
    to_date="2025-08-06"
)
brunswick_df = pd.DataFrame(brunswick_data)
print(f"Brunswick vessels found: {len(brunswick_df)}")
brunswick_df.head()

In [ ]:
# Fetch inradius history for Bremerhaven area
bremerhaven_data = fetch_inradius_history(
    lat=53.5542,
    lon=8.56048,
    radius=10,
    from_date="2025-07-31",
    to_date="2025-08-06"
)
bremerhaven_df = pd.DataFrame(bremerhaven_data)
print(f"Bremerhaven vessels found: {len(bremerhaven_df)}")
bremerhaven_df.head()

In [ ]:
# Get common ships that were at both Bremerhaven and Brunswick
common_uuids = bremerhaven_df[bremerhaven_df['uuid'].isin(brunswick_df['uuid'])]

In [ ]:
# Get all departure points from Bremerhaven
departure_entries = []
bremerhaven_names = ['DE BRV', 'DEBRV', 'BREMERHAVEN', 'Bremerhaven']

for uuid in common_uuids['uuid'].unique():
    ship_df = common_uuids[common_uuids['uuid'] == uuid].sort_values('last_pos_utc', ascending=True).reset_index(drop=True)
    for i in range(1, len(ship_df)):
        prev_dest = ship_df.loc[i-1, 'destination']
        curr_dest = ship_df.loc[i, 'destination']
        # Find the pivot: destination changes from Bremerhaven to something else
        if prev_dest in bremerhaven_names and curr_dest not in bremerhaven_names:
            # Now look for the first entry with speed != 0.0 for the new destination
            last_zero_speed_idx = None
            for j in range(i, len(ship_df)):
                if ship_df.loc[j, 'destination'] == curr_dest:
                    if ship_df.loc[j, 'speed'] == 0.0:
                        last_zero_speed_idx = j
                    elif ship_df.loc[j, 'speed'] != 0.0 and last_zero_speed_idx is not None:
                        departure_entries.append(ship_df.loc[last_zero_speed_idx])
                        break

departure_df = pd.DataFrame(departure_entries)
departure_df = departure_df.drop(columns=["last_pos_epoch", "distance_nm"])
departure_df = departure_df.reset_index(drop=True)

In [ ]:
# Get the coordinates of the journeys for the departures
journey_positions = []

for journey_id, (idx, row) in enumerate(tqdm(departure_df.iterrows(), total=len(departure_df), desc="Fetching 2mo routes")):
    uuid = row["uuid"]
    from_date = row["last_pos_utc"]
    # Add the departure point (speed is 0)
    journey_positions.append({
        "uuid": uuid,
        "journey_id": journey_id,
        "lat": row["lat"],
        "lon": row["lon"],
        "speed": row["speed"],
        "course": row.get("course", None),
        "heading": row.get("heading", None),
        "destination": row.get("destination", None),
        "last_position_UTC": row["last_pos_utc"],
        "no_arrival_in_brunswick": None  # Will be set below
    })
    # Fetch both months
    positions = fetch_positions(uuid, from_date, 0) + fetch_positions(uuid, from_date, 1)
    positions = sorted(positions, key=lambda p: p["last_position_UTC"])
    arrived = False
    for p in positions:
        lat, lon, speed = p["lat"], p["lon"], p["speed"]
        # Check if in Brunswick area and stopped
        if (31.0 <= lat <= 31.2) and (-81.6 <= lon <= -81.5) and speed == 0.0:
            # Add the arrival point as well
            journey_positions.append({
                "uuid": uuid,
                "journey_id": journey_id,
                "lat": lat,
                "lon": lon,
                "speed": speed,
                "course": p.get("course"),
                "heading": p.get("heading"),
                "destination": p.get("destination"),
                "last_position_UTC": p.get("last_position_UTC"),
                "no_arrival_in_brunswick": None  # Will be set below
            })
            arrived = True
            break
        # Only keep positions not in Brunswick area and moving
        if not ((31.0 <= lat <= 31.2) and (-81.6 <= lon <= -81.5)) and speed != 0.0:
            journey_positions.append({
                "uuid": uuid,
                "journey_id": journey_id,
                "lat": lat,
                "lon": lon,
                "speed": speed,
                "course": p.get("course"),
                "heading": p.get("heading"),
                "destination": p.get("destination"),
                "last_position_UTC": p.get("last_position_UTC"),
                "no_arrival_in_brunswick": None  # Will be set below
            })
    # If not arrived, mark all journey points for this journey_id
    if not arrived:
        for jp in journey_positions:
            if jp["journey_id"] == journey_id:
                jp["no_arrival_in_brunswick"] = True
                
journey_positions_df = pd.DataFrame(journey_positions)
journey_positions_df = journey_positions_df.sort_values(by=["journey_id", "last_position_UTC"], ascending=[True, True]).reset_index(drop=True)

In [ ]:
# Keep only journeys that arrive in Brunswick
arrived_journey_ids = journey_positions_df[journey_positions_df["no_arrival_in_brunswick"] != True]["journey_id"].unique()
arrived_df = journey_positions_df[journey_positions_df["journey_id"].isin(arrived_journey_ids)].reset_index(drop=True)
arrived_df["last_position_UTC"] = arrived_df["last_position_UTC"].apply(utc_to_local_str)
arrived_df = arrived_df.rename(columns={"last_position_UTC": "timestamp"})
arrived_df['timestamp'] = pd.to_datetime(arrived_df['timestamp'])
arrived_df = arrived_df.drop(columns=["no_arrival_in_brunswick"]) # arrived journeys positions

In [ ]:
# Extract start and end points of each journey
start_end_rows = []
for journey_id, group in tqdm(arrived_df.groupby('journey_id'), desc="Processing journeys"):
    first_row = group.iloc[0]
    last_row = group.iloc[-1]
    start_end_rows.append(first_row)
    start_end_rows.append(last_row)

start_end_journeys = pd.DataFrame(start_end_rows)
start_end_journeys.to_csv('start_end_journeys.csv', index=False)

### Get vessel info

In [ ]:
# Get unique uuids from df
unique_uuids = start_end_journeys["uuid"].unique()

vessel_infos = []
for uuid in unique_uuids:
    info = fetch_vessel_info(uuid)
    if info:
        vessel_infos.append(info)

# Convert to DataFrame and save to CSV
vessel_info_df = pd.DataFrame(vessel_infos)
vessel_info_df.to_csv("vessel_info.csv", index=False)